# Capstone 03 — Synced (read) tables + Staging (write) tables

Two sets of Lakebase objects:

1. **Synced tables** — auto-refreshed copies of UC Delta tables. The app
   reads these for sub-10ms latency.
   - `customers_synced`     (CONTINUOUS — live updates)
   - `transactions_synced`  (CONTINUOUS — recent activity feed)
   - `products_synced`      (TRIGGERED hourly — slow-changing)

2. **Staging tables** — written by the app via psycopg2. Forward-ETL job
   reads these and MERGEs into Delta gold.
   - `customer_notes_staging`
   - `customer_segment_overrides_staging`
   - `customer_audit_log`

Run AFTER notebook 02.

In [0]:
%pip install --quiet --upgrade databricks-sdk psycopg2-binary

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "oneenv_lakebase_capstone_catalog", "Source UC catalog (gold tables)")
dbutils.widgets.text("schema", "gold", "Source schema")
dbutils.widgets.text("uc_lakebase_catalog", "oneenv_lakebase_capstone_lb_catalog", "UC catalog backed by Lakebase")
dbutils.widgets.text("instance_name", "capstone-pg", "Lakebase instance name")
dbutils.widgets.text("database_name", "capstone_db", "Postgres database name")
dbutils.widgets.text("storage_catalog", "oneenv_lakebase_capstone_catalog", "Catalog for sync pipeline storage")
dbutils.widgets.text("storage_schema", "pipelines", "Schema for sync pipeline storage")

CAT = dbutils.widgets.get("catalog")
SCH = dbutils.widgets.get("schema")
LB_CAT = dbutils.widgets.get("uc_lakebase_catalog")
INSTANCE = dbutils.widgets.get("instance_name")
DB_NAME = dbutils.widgets.get("database_name")
STORAGE_CAT = dbutils.widgets.get("storage_catalog")
STORAGE_SCH = dbutils.widgets.get("storage_schema")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {STORAGE_CAT}.{STORAGE_SCH}")

DataFrame[]

## 1. Create synced tables (reverse ETL)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import AlreadyExists
from databricks.sdk.service.database import (
    SyncedDatabaseTable, SyncedTableSpec, NewPipelineSpec,
    SyncedTableSchedulingPolicy,
)

w = WorkspaceClient()


def sync(name: str, source_table: str, mode: SyncedTableSchedulingPolicy, pk: list[str]):
    full_name = f"{LB_CAT}.public.{name}"
    # If synced table already exists in SDK metadata, skip
    try:
        existing = w.database.get_synced_database_table(name=full_name)
        print(f"  [skip] {full_name} already exists (state={existing.data_synchronization_status})")
        return existing
    except Exception:
        pass
    # Create new synced table; skip if destination already exists in Postgres
    spec = SyncedTableSpec(
        source_table_full_name=source_table,
        primary_key_columns=pk,
        scheduling_policy=mode,
        new_pipeline_spec=NewPipelineSpec(
            storage_catalog=STORAGE_CAT,
            storage_schema=STORAGE_SCH,
        ),
    )
    try:
        result = w.database.create_synced_database_table(
            SyncedDatabaseTable(name=full_name, spec=spec)
        )
        print(f"  [created] {full_name}")
        return result
    except AlreadyExists:
        print(f"  [skip] {full_name} already exists in Postgres")
        return None

sync("customers_synced",
     f"{CAT}.{SCH}.customers",
     SyncedTableSchedulingPolicy.CONTINUOUS,
     ["customer_id"])
sync("transactions_synced",
     f"{CAT}.{SCH}.transactions",
     SyncedTableSchedulingPolicy.CONTINUOUS,
     ["transaction_id"])
sync("products_synced",
     f"{CAT}.{SCH}.products",
     SyncedTableSchedulingPolicy.TRIGGERED,
     ["product_id"])

print("Synced tables submitted. Initial sync may take a few minutes.")

  [skip] oneenv_lakebase_capstone_lb_catalog.public.customers_synced already exists in Postgres
  [skip] oneenv_lakebase_capstone_lb_catalog.public.transactions_synced already exists in Postgres
  [skip] oneenv_lakebase_capstone_lb_catalog.public.products_synced already exists in Postgres
Synced tables submitted. Initial sync may take a few minutes.


## 2. Create staging tables in Lakebase via psycopg2

Uses an OAuth token (current user identity) to connect.

In [0]:
import psycopg2

ENDPOINT_NAME = f"projects/{INSTANCE}/branches/production/endpoints/primary"
endpoint = w.postgres.get_endpoint(name=ENDPOINT_NAME)
host = endpoint.status.hosts.host
me = w.current_user.me().user_name
token = w.postgres.generate_database_credential(endpoint=ENDPOINT_NAME).token

conn = psycopg2.connect(
    host=host, port=5432, dbname=DB_NAME,
    user=me, password=token, sslmode="require",
)
conn.autocommit = True
cur = conn.cursor()

DDL = """
CREATE EXTENSION IF NOT EXISTS pgcrypto;

CREATE TABLE IF NOT EXISTS customer_notes_staging (
    note_id        UUID         PRIMARY KEY DEFAULT gen_random_uuid(),
    customer_id    VARCHAR(20)  NOT NULL,
    author_email   VARCHAR(200) NOT NULL,
    note_text      TEXT         NOT NULL,
    sentiment      REAL,
    created_at     TIMESTAMPTZ  NOT NULL DEFAULT NOW(),
    processed      BOOLEAN      NOT NULL DEFAULT FALSE,
    processed_at   TIMESTAMPTZ
);
CREATE INDEX IF NOT EXISTS idx_notes_customer ON customer_notes_staging (customer_id);
CREATE INDEX IF NOT EXISTS idx_notes_unprocessed ON customer_notes_staging (processed) WHERE processed = FALSE;

CREATE TABLE IF NOT EXISTS customer_segment_overrides_staging (
    override_id      UUID         PRIMARY KEY DEFAULT gen_random_uuid(),
    customer_id      VARCHAR(20)  NOT NULL UNIQUE,
    override_segment VARCHAR(10)  NOT NULL,
    reason           TEXT,
    author_email     VARCHAR(200) NOT NULL,
    created_at       TIMESTAMPTZ  NOT NULL DEFAULT NOW(),
    processed        BOOLEAN      NOT NULL DEFAULT FALSE,
    processed_at     TIMESTAMPTZ
);

CREATE TABLE IF NOT EXISTS customer_audit_log (
    audit_id     BIGSERIAL    PRIMARY KEY,
    customer_id  VARCHAR(20)  NOT NULL,
    action       VARCHAR(50)  NOT NULL,
    actor_email  VARCHAR(200) NOT NULL,
    payload      JSONB,
    created_at   TIMESTAMPTZ  NOT NULL DEFAULT NOW()
);
CREATE INDEX IF NOT EXISTS idx_audit_customer ON customer_audit_log (customer_id);
"""
cur.execute(DDL)

# Show the resulting object list
cur.execute("""
    SELECT schemaname, tablename FROM pg_tables
    WHERE schemaname = 'public' ORDER BY tablename
""")
for s, t in cur.fetchall():
    print(f"  {s}.{t}")

cur.close()
conn.close()

  public.customer_audit_log
  public.customer_notes_staging
  public.customer_segment_overrides_staging
  public.customers_synced
  public.products_synced
  public.transactions_synced


In [0]:
print("Lakebase schema ready.")
print(f"PG_HOST={host}")
print(f"PG_DATABASE={DB_NAME}")

Lakebase schema ready.
PG_HOST=ep-young-dust-d2d83gtg.database.us-east-1.cloud.databricks.com
PG_DATABASE=capstone_db
